In [1]:
#|default_exp dataset

In [2]:
#| export
from pathlib import Path
import torch
import os
import glob
import numpy as np
import torch
import pickle
from torch.utils.data import Dataset, DataLoader

In [3]:
#| export
def good_luck():
    return True

In [4]:
#| export
def load_graphs(graph_path):
    with open(os.path.join(graph_path,'graphs.pickle'), 'rb') as f:
        graphs = pickle.load(f)
    return graphs
    
    
    
class TPUDatasetV0(Dataset):
    def __init__(self, files, path, nmax=-1, data_id=0):
        self.files, self.graphs = files,load_graphs(path)
        self.nmax, self.data_id = nmax,data_id
        self.path = path
        
    def __len__(self):
        return len(self.files)
        
    def __getitem__(self, idx):
        path = self.files[idx]
        idx = '/'.join(path.replace(str(self.path),'').split('/')[:-1])
        graph = self.graphs[idx[1:]]
        data = np.load(path)
        
        node_opcode = torch.from_numpy(graph['node_opcode']).long() - 1 # start from 1
        edge_index = torch.from_numpy(graph['edge_index']).long()
        #print(torch.from_numpy(graph['node_feat']).min())
        node_feat = (3.0 + torch.from_numpy(graph['node_feat'])).log() #min = -2
        node_config_ids = torch.from_numpy(graph['node_config_ids']).long()
        
        node_config_feat = torch.from_numpy(data['node_config_feat']).long() + 1 #start from -1
        config_runtime = 0.001*torch.from_numpy(data['config_runtime'])/node_config_feat.shape[0]
        #config_runtime_norm = torch.from_numpy(data['config_runtime_norm']).float()
        #config_runtime = (torch.from_numpy(data['runtime'])).log()
        
        n = config_runtime.shape[0]
        if self.nmax > 0:
            perm = torch.randperm(n)[:self.nmax]
            config_runtime = config_runtime[perm]
            node_config_feat = node_config_feat[perm]
            #config_runtime_norm = config_runtime_norm[perm]
            
        return {'node_feat':node_feat, 'node_opcode':node_opcode, 'edge_index':edge_index,'node_config_feat':node_config_feat,
                'node_config_ids':node_config_ids, 
                'config_runtime':config_runtime}, \
               {'data_idx':self.data_id, 'config_runtime':config_runtime}

In [5]:
class CONFIG:
    path = Path('/opt/slh/tpuoptimize/data/layout_converted_z2_1000')
    nmax_t=256
    nmax_v=-1
    

    

In [6]:
ds_t = torch.utils.data.ConcatDataset([
  TPUDatasetV0(list(glob.glob(os.path.join(CONFIG.path,'nlp','default','train','**/*.npz'), 
                            recursive=True)),nmax=CONFIG.nmax_t,data_id=0, path = CONFIG.path),
  TPUDatasetV0(list(glob.glob(os.path.join(CONFIG.path,'nlp','random','train','**/*.npz'), 
                            recursive=True)),nmax=CONFIG.nmax_t,data_id=1, path = CONFIG.path),
  TPUDatasetV0(list(glob.glob(os.path.join(CONFIG.path,'xla','default','train','**/*.npz'), 
                            recursive=True)),nmax=CONFIG.nmax_t,data_id=2, path = CONFIG.path),
  TPUDatasetV0(list(glob.glob(os.path.join(CONFIG.path,'xla','random','train','**/*.npz'), 
                            recursive=True)),nmax=CONFIG.nmax_t,data_id=3, path = CONFIG.path),
])

dl = DataLoader(ds_t, batch_size=1, shuffle=True, num_workers=0)
data = next(iter(dl))
torch.save(data, 'batch.pt')


In [7]:
#|hide
#|eval: false
from nbdev.doclinks import nbdev_export
nbdev_export()